In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install mne --quiet

import mne
import os, gc, numpy as np, pandas as pd, scipy.signal as sig
# from scipy.signal import hilbert
import matplotlib.pyplot as plt

In [ ]:
# Set path
root_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/EEGData/'
subject_dirs = sorted([d for d in os.listdir(root_dir) if d.isdigit()])
print(subject_dirs)

In [ ]:
for subj in subject_dirs:
  subj_path = os.path.join(root_dir, subj)
  session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('.set')])

  for session_file in session_files:
    session_path = os.path.join(subj_path, session_file)

    # Load
    raw = mne.io.read_raw_eeglab(session_path, preload=True)
    print(f"Processing: {session_path}")

    # Set standard montage
    raw.set_montage('GSN-HydroCel-128')

    # Re-referencing
    try:
      raw.set_eeg_reference(ref_channels=['E57', 'E100'])
      print(f"{subj}: Re-referenced using LM/RM average.")
    except Exception as e:
      print(f"{subj}: Error during referencing — {e}")

    raw.filter(0.1, None) # High-pass filter because RuntimeWarning: The data
    # has not been high-pass filtered. For good ICA performance, it should be
    # high-pass filtered (e.g., with a 1.0 Hz lower bound) before fitting ICA.

    # ICA on re-referenced, high-passed filtered EEG
    try:
      ica_raw = mne.preprocessing.ICA(random_state=97, method='fastica')
      ica_raw.fit(raw)
      print(f"{subj}: ICA on raw EEG:")
      ica_raw.plot_components(picks=range(15), inst=raw, cmap='interactive')
    except Exception as e:
      print(f"{subj}: Error during ICA_raw — {e}")

In [ ]:
for subj in subject_dirs:
  subj_path = os.path.join(root_dir, subj)
  session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('.set')])

  for session_file in session_files:
    session_path = os.path.join(subj_path, session_file)

    # Load
    raw = mne.io.read_raw_eeglab(session_path, preload=True)
    print(f"Processing: {session_path}")

    # # Crop to reduce memory usage
    # try:
    #   raw.crop(tmax=1800)  # Use only the first 30 minutes
    # except Exception as e:
    #   print(f"{subj}: Error during cropping - {e}")

    # Set standard montage
    raw.set_montage('GSN-HydroCel-128')

    # Re-referencing
    try:
      raw.set_eeg_reference(ref_channels=['E57', 'E100'])
      print(f"{subj}: Re-referenced using LM/RM average.")
    except Exception as e:
      print(f"{subj}: Error during referencing - {e}")

    # ICA on re-referenced, filtered EEG
    try:
      raw.filter(0.1, None) # High pass filtered
      raw.notch_filter(60)  # Notch filter
    except Exception as e:
      print(f"{subj}: Error during filtering - {e}")

    # make sure in order of variance explained
    try:
      ica_filtered = mne.preprocessing.ICA(n_components=15, random_state=97, method='fastica')
      ica_filtered.fit(raw)
    except Exception as e:
      print(f"{subj}: Error during ICA fitting - {e}")

    # Print variance explained by first 15 components
    try:
      # Sort components by explained variance
      explained_var = ica_filtered.get_explained_variance_ratio(raw)
      top15_indices = np.argsort(explained_var)[::-1][:15]
      print(f"{subj}: Total variance (top 15 comps): {np.sum(explained_var[top15_indices]) * 100:.2f}%")
    except Exception as e:
      print(f"{subj}: Error during variance calculation - {e}")

    # ICA plots
    try:
      print(f"{subj}: ICA topomap on filtered EEG:")
      ica_filtered.plot_components(inst=raw)
      print(f"{subj}: ICA component time series plot:")
      ica_filtered.plot_sources(raw)
    except Exception as e:
      print(f"{subj}: Error during ICA plotting - {e}")